In [ ]:
import pandas as pd
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import config as cf
import my_secrets as sc

#!pip install 'accelerate>=0.26.0' 

In [ ]:
if not os.path.exists(cf.HUGGING_CACHE):
    os.mkdir(cf.HUGGING_CACHE)

os.environ["TRANSFORMERS_CACHE"] = cf.HUGGING_CACHE
os.environ["HF_HOME"] = cf.HUGGING_CACHE

In [ ]:
from huggingface_hub import login
login(token=sc.HUGGING_TOKEN_NOTEBOOK)

In [ ]:
model_id = "mistralai/Mistral-7B-Instruct-v0.1"

kwargs = {
     "torch_dtype": torch.bfloat16,
     "device_map": "cuda:0",  
}

model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=model_id, **kwargs
)
tokenizer = AutoTokenizer.from_pretrained(model_id, **kwargs)

In [ ]:
import re

USING_CONTEXT = "Using this information: {context} " + "answer the Question: {prompt}"

def format_prompt(prompt, context=None, system=""):
    system_msg = f"<<SYS>> {system} <</SYS>>"
    instruction = USING_CONTEXT.format(context=context, prompt=prompt)
    formatted_prompt = f"<s>[INST] {system_msg} {instruction} [/INST]"
    return formatted_prompt

def prompt_open_model(prompt, model, tokenizer, max_tokens=200):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to("cuda:0")
    attention_mask = torch.ones(input_ids.shape).to("cuda:0")

    output = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_length=input_ids.shape[1] + max_tokens,
        do_sample=True,
        temperature=0.8,
        top_p=0.65,
        # top_k=25,
        num_return_sequences=1,
        # num_beams=2,
        no_repeat_ngram_size=3,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

    response = tokenizer.decode(output[0], skip_special_tokens=True)
    answer = response.removeprefix(prompt.strip("<s>")).strip("\n")
    return answer

In [ ]:
system = 'You are a helpful AI assistent which helps to reply to citizens who want to object against a parking fine'
context = 'Nou ik wilde dus naar een vriendin van mijn gaan met kerst. Zij zegt tegen mij: met al die meuk die mee moet, pak je beter gewoon even de auto. Ik heb van die bezoekersdingen, dan is het niet zo stervensduur. Dus ik geef haar mijn kenteken, maar ik ben moe weet je dus ik geeft een Z door in plaats van een X, die juist wel in mijn kenteken zit. Zij vult het keurig netjes in, maarja wel fout dus, maar dat wist ik toen nog niet. Paar weken later, dikke boete op de mat! Die wil ik natuurlijk niet betalen!'
prompt = 'Give a relevant response (in Dutch) in which you match the tone of voice of this citizen' 

formatted_prompt = format_prompt(prompt,  context, system)
#formatted_prompt = "Ik geef je zo een fictionele tekst van een inwoner die bezwaar wil maken voor een boete. Geef een relevante reactie, met hetzelfde taalgebruik als die van de inwoner. Tekst: 'Nou ik wilde dus naar een vriendin van mijn gaan met kerst. Zij zegt tegen mij: met al die meuk die mee moet, pak je beter gewoon even de auto. Ik heb van die bezoekersdingen, dan is het niet zo stervensduur. Dus ik geef haar mijn kenteken, maar ik ben moe weet je dus ik geeft een Z door in plaats van een X, die juist wel in mijn kenteken zit. Zij vult het keurig netjes in, maarja wel fout dus, maar dat wist ik toen nog niet. Paar weken later, dikke boete op de mat! Die wil ik natuurlijk niet betalen!"
#formatted_prompt = "Je krijgt zo een fictioneel bericht van een inwoner die bezwaar wil maken tegen een boete. Geef een passende reactie en spiegel daarin het taalgebruik van de inwoner. Tekst: 'Dit is echt ongelofelijk! Ik had in de Kinkerstraat eindelijk mijn Fiat Panda tussen twee Tesla’s gewurmd, ga ik naar die automaat om 6 euro per uur weg te pissen, zie ik zo’n scanauto langsrijden! Ik zei al tegen mijn vriend, dit gaat niet goed hoor. Deze ziet ons staan, maar we zijn nog met ons passie bezig. En inderdaad onze plaat AB-09-CC stond er nog niet in, want een paar weken later lag die bon op mijn mat.'"

In [ ]:
answer = prompt_open_model(formatted_prompt, model, tokenizer)
print(answer)